# Tensor Parallelism

# Prepare data

$$
y = xw
$$

x(n, dim)

w(dim, 2dim)

y(n, 2dim)

y_label(n, 2dim)

gradient(dim, 2dim)


In [42]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [43]:
bs = 2 
row = dim = 4
col = out_dim = dim * 2

x = torch.arange(bs * dim, dtype = torch.float32).reshape(bs, dim) 

w = torch.arange(dim * out_dim, dtype = torch.float32, requires_grad = True).reshape(dim, out_dim)

print(x)
print(w)

tensor([[0., 1., 2., 3.],
        [4., 5., 6., 7.]])

tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11., 12., 13., 14., 15.],
        [16., 17., 18., 19., 20., 21., 22., 23.],
        [24., 25., 26., 27., 28., 29., 30., 31.]], grad_fn=<ViewBackward0>)

In [44]:
y_label = torch.randn(bs, out_dim)
y_pred = torch.randn(bs, out_dim)

# pytorch gradient

先采用pytorch自动求导计算梯度。可以用来检验我们手动求导的准确性

In [45]:
w.retain_grad()
w.grad = None
mse_loss = nn.MSELoss(reduction='mean')

y_pred = x @ w
loss_torch = mse_loss(y_pred, y_label)
loss_torch.backward()
print(loss_torch)
print(w.grad)

tensor(82899.1797, grad_fn=<MseLossBackward0>)

tensor([[152.8711, 162.3157, 174.0892, 185.1861, 196.1222, 206.9512, 218.3441,
         228.8488],
        [205.0980, 217.6793, 233.2117, 247.7463, 262.1614, 276.5011, 291.4515,
         305.4002],
        [257.3250, 273.0429, 292.3341, 310.3064, 328.2006, 346.0509, 364.5589,
         381.9516],
        [309.5518, 328.4066, 351.4566, 372.8666, 394.2398, 415.6008, 437.6664,
         458.5030]])

# tensor gradient

$$
\begin{align}
y &= xw \\
\mathcal{l} &=||y'-y||^2 = ||y'-xw||^2 \\
w &= \arg\min_{w}||y'-xw||^2 \\
\frac{\partial l}{\partial w} &=\frac{\partial l}{\partial y}\frac{\partial y}{\partial w} =  2x^T(y'-xw)\\
\end{align}
$$

In [46]:
y_pred = x @ w
delta_y = y_pred - y_label

In [47]:
delta_w = x.t() @ delta_y / out_dim # dim
print(delta_w)

tensor([[152.8711, 162.3157, 174.0892, 185.1861, 196.1222, 206.9512, 218.3441,
         228.8488],
        [205.0980, 217.6793, 233.2117, 247.7463, 262.1614, 276.5011, 291.4515,
         305.4002],
        [257.3250, 273.0429, 292.3341, 310.3064, 328.2006, 346.0509, 364.5589,
         381.9516],
        [309.5518, 328.4066, 351.4566, 372.8666, 394.2398, 415.6008, 437.6664,
         458.5030]], grad_fn=<DivBackward0>)

In [48]:
w.retain_grad()
w.grad = None
mse_loss = nn.MSELoss(reduction='mean')

y_pred = x @ w
loss_torch = mse_loss(y_pred, y_label)
loss_torch.backward()
print(loss_torch)
print(w.grad)

tensor(82899.1797, grad_fn=<MseLossBackward0>)

tensor([[152.8711, 162.3157, 174.0892, 185.1861, 196.1222, 206.9512, 218.3441,
         228.8488],
        [205.0980, 217.6793, 233.2117, 247.7463, 262.1614, 276.5011, 291.4515,
         305.4002],
        [257.3250, 273.0429, 292.3341, 310.3064, 328.2006, 346.0509, 364.5589,
         381.9516],
        [309.5518, 328.4066, 351.4566, 372.8666, 394.2398, 415.6008, 437.6664,
         458.5030]])

# Tensor Parallel

In [49]:
import copy
# w.grad = None
w_row = torch.arange(dim * out_dim, dtype = torch.float32).reshape(dim, out_dim)
w_col = torch.arange(dim * out_dim, dtype = torch.float32).reshape(dim, out_dim)

## row-tensor parallel

In [50]:
row_dim = dim // 2
w_row_1 = w_row[:row_dim, :]
w_row_2 = w_row[row_dim:, :]

print(w_row_1.shape)
print(w_row_2.shape)
print(w_row_1)
print(w_row_2)

torch.Size([2, 8])

torch.Size([2, 8])

tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11., 12., 13., 14., 15.]])

tensor([[16., 17., 18., 19., 20., 21., 22., 23.],
        [24., 25., 26., 27., 28., 29., 30., 31.]])

In [51]:
x_col_1 = x[:, :row_dim]
x_col_2 = x[:, row_dim:]
y_1 = x[:, :row_dim] @ w_row_1
y_2 = x[:, row_dim:] @ w_row_2
print(y_1.shape)
print(y_2.shape)

torch.Size([2, 8])

torch.Size([2, 8])

In [52]:
delta_y = (y_1 + y_2 - y_label)

grad_row_1 = x_col_1.t() @ delta_y 
grad_row_2 = x_col_2.t() @ delta_y 

print(grad_row_1.shape)
print(grad_row_2.shape)

torch.Size([2, 8])

torch.Size([2, 8])

In [53]:
grad_row = torch.cat((grad_row_1,grad_row_2), dim = 0) / out_dim
print(grad_row)

tensor([[152.8711, 162.3157, 174.0892, 185.1861, 196.1222, 206.9512, 218.3441,
         228.8488],
        [205.0980, 217.6793, 233.2117, 247.7463, 262.1614, 276.5011, 291.4515,
         305.4002],
        [257.3250, 273.0429, 292.3341, 310.3064, 328.2006, 346.0509, 364.5589,
         381.9516],
        [309.5518, 328.4066, 351.4566, 372.8666, 394.2398, 415.6008, 437.6664,
         458.5030]])

## col-tensor parallel

In [54]:
col_dim = out_dim // 2
w_col_1 = w_col[: , :col_dim]
w_col_2 = w_col[: , col_dim:]

print(w_col_1.shape)
print(w_col_2.shape)
print(w_col_1)
print(w_col_2)

torch.Size([4, 4])

torch.Size([4, 4])

tensor([[ 0.,  1.,  2.,  3.],
        [ 8.,  9., 10., 11.],
        [16., 17., 18., 19.],
        [24., 25., 26., 27.]])

tensor([[ 4.,  5.,  6.,  7.],
        [12., 13., 14., 15.],
        [20., 21., 22., 23.],
        [28., 29., 30., 31.]])

In [55]:
y_1 = x @ w_col_1
y_2 = x @ w_col_2
print(y_1.shape)
print(y_2.shape)

torch.Size([2, 4])

torch.Size([2, 4])

In [56]:
y_1_delta = y_1 - y_label[:, col_dim:]
y_2_delta = y_2 - y_label[:, :col_dim]
print(y_1_delta.shape)
print(y_2_delta.shape)

torch.Size([2, 4])

torch.Size([2, 4])

In [57]:
grad_col_1 = x.t() @ y_1_delta
grad_col_2 = x.t() @ y_2_delta 
grad_col = torch.cat( (grad_col_1, grad_col_2), dim = 1) / out_dim
print(grad_col_1.shape)
print(grad_col_2.shape)
print(grad_col.shape)
print(grad_col_1)
print(grad_col_2)
print(grad_col)

torch.Size([4, 4])

torch.Size([4, 4])

torch.Size([4, 8])

tensor([[1216.9775, 1303.6099, 1394.7524, 1478.7900],
        [1633.2911, 1748.0087, 1867.6119, 1979.2014],
        [2049.6047, 2192.4075, 2340.4714, 2479.6128],
        [2465.9182, 2636.8064, 2813.3308, 2980.0242]])

tensor([[1574.9691, 1650.5253, 1744.7135, 1833.4890],
        [2104.7844, 2205.4343, 2329.6931, 2445.9702],
        [2634.5996, 2760.3433, 2914.6729, 3058.4514],
        [3164.4148, 3315.2522, 3499.6526, 3670.9326]])

tensor([[152.1222, 162.9512, 174.3441, 184.8488, 196.8711, 206.3157, 218.0892,
         229.1861],
        [204.1614, 218.5011, 233.4515, 247.4002, 263.0981, 275.6793, 291.2116,
         305.7463],
        [256.2006, 274.0509, 292.5589, 309.9516, 329.3250, 345.0429, 364.3341,
         382.3064],
        [308.2398, 329.6008, 351.6664, 372.5030, 395.5518, 414.4065, 437.4566,
         458.8666]])